<a href="https://colab.research.google.com/github/justorfc/Est_Python_R_2026_1/blob/main/11_Semana_11_Sesion_01_Fundamentos_Diagnostico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Llegamos a una sesión que separa a los verdaderos analistas de datos de aquellos que solo aplican fórmulas a ciegas.

Un modelo puede tener un $R^2$ del 99%, pero si viola los supuestos, sus predicciones no sirven para tomar decisiones de ingeniería.

### 📘 Estructura del Notebook: Semana 11 - Sesión 1

**Título del Notebook:** `Semana_11_Sesion_01_Fundamentos_Diagnostico.ipynb`

#### **Celda 1: Markdown (Encabezado y Fase 1: IA)**

# SEMANA 11: Diagnóstico Avanzado del Modelo Lineal
**Asignatura:** Estadística Aplicada con Python y R
**Programas:** Ingeniería Agrícola, Agroindustrial y Civil

## Primera Hora: Actividad "Estudia y Aprende"
Ajustar una línea a un conjunto de puntos es fácil. Lo difícil es saber si esa línea es confiable. En ingeniería, si el modelo está mal diagnosticado, el puente se cae o la cosecha se pierde. Antes de programar, interactúa con tu IA para dominar la teoría detrás del diagnóstico.

**PROMPT 1 - INICIO:**
> Actúa como tutor experto en diagnóstico de modelos de regresión.
> Tema: Diagnóstico avanzado del modelo lineal.
> 1. Explica por qué un modelo debe diagnosticarse.
> 2. Explica qué son residuos y cómo se interpretan.
> 3. Explica homocedasticidad vs heterocedasticidad.
> 4. Explica qué es leverage.
> 5. Explica qué es distancia de Cook.
> 6. Explica qué sucede si los supuestos se violan.
> Hazme 3 preguntas para verificar comprensión y corrige mis respuestas.

#### **Celda 2: Markdown (Fase 2: Orientación Docente - La anatomía del diagnóstico)**

## Segunda Hora: Orientación Docente en Python
La regresión lineal asume que los errores ($\epsilon$) se comportan de forma ideal. El diagnóstico nos sirve para cazar "patologías" en nuestros datos:

1. **Heterocedasticidad:** La varianza de los errores no es constante. (Ej. El modelo predice muy bien resistencias bajas, pero se equivoca por mucho en resistencias altas).
2. **Falta de Normalidad:** Los errores no siguen una campana de Gauss. Esto arruina los Valores-p y los intervalos de confianza.
3. **Observaciones Influyentes (Leverage y Cook):** Un solo dato anómalo (como un error de calibración del sensor) está jalando toda la línea de regresión hacia él, distorsionando el modelo entero.

**La Premisa:** Un modelo con buen $R^2$ puede ser estadísticamente inválido.

#### **Celda 3: Código Python (Simulación de un Modelo "Enfermo")**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

sns.set_theme(style="whitegrid")
np.random.seed(100)

# ==========================================
# 1. CREACIÓN DE UN MODELO CON HETEROCEDASTICIDAD Y OUTLIERS
# ==========================================
# Contexto: Presión aplicada a una prensa (X) vs Deformación del material (Y)
presion_X = np.linspace(10, 100, 50)

# El error aumenta a medida que aumenta la presión (Heterocedasticidad)
error_creciente = np.random.normal(0, presion_X * 0.15, 50)
deformacion_Y = 5.0 + 2.5 * presion_X + error_creciente

# Introducimos una observación influyente artificial (Leverage alto + Gran error)
presion_X[49] = 150 # Dato muy alejado en X (Leverage)
deformacion_Y[49] = 100 # Dato anómalo en Y (jala el modelo hacia abajo)

df_prensa = pd.DataFrame({'Presion': presion_X, 'Deformacion': deformacion_Y})

# Ajustamos el modelo (sabiendo que los datos están "enfermos")
X_const = sm.add_constant(df_prensa['Presion'])
modelo_enfermo = sm.OLS(df_prensa['Deformacion'], X_const).fit()

# Extraemos los residuos
residuos = modelo_enfermo.resid
predicciones = modelo_enfermo.fittedvalues

print(modelo_enfermo.summary().tables[0])
print("\nNota el R-cuadrado alto. ¡Pero veamos los residuos!")

#### **Celda 4: Código Python (Diagnóstico Visual de la Heterocedasticidad)**

In [ ]:
# ==========================================
# 2. ANÁLISIS VISUAL DE LOS RESIDUOS
# ==========================================
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: El Modelo vs Los Datos (Observa el punto anómalo a la derecha)
ax[0].scatter(df_prensa['Presion'], df_prensa['Deformacion'], color='navy', label='Datos')
ax[0].plot(df_prensa['Presion'], predicciones, color='red', linewidth=2, label='Ajuste Lineal')
ax[0].set_title('Modelo Ajustado (Presión vs Deformación)')
ax[0].set_xlabel('Presión Aplicada')
ax[0].set_ylabel('Deformación Observada')
ax[0].legend()

# Gráfico 2: Residuos vs Valores Ajustados (La prueba de fuego)
ax[1].scatter(predicciones, residuos, color='darkorange')
ax[1].axhline(0, color='black', linestyle='--')
ax[1].set_title('Residuos vs Predicciones (Heterocedasticidad)')
ax[1].set_xlabel('Deformación Predicha')
ax[1].set_ylabel('Residuos (Error)')

plt.tight_layout()
plt.show()

# Interpretación Ingenieril:
# Observa el Gráfico 2. En lugar de ver una nube aleatoria uniforme, los puntos forman
# un "megáfono" o "embudo" que se abre hacia la derecha.
# Esto significa que cuanto mayor es la presión, más grande es el error de nuestro modelo.
# Además, el punto influyente (abajo a la derecha) distorsionó la línea de tendencia.

#### **Celda 5: Markdown (Transición a R y RMarkdown)**

## Trabajo Autónomo: Transición a R y RMarkdown

R es el lenguaje por excelencia para el diagnóstico de modelos lineales. Sus gráficos por defecto ya incluyen el análisis de influencia.

**Paso 1: Validación en Colab (Entorno R)**
1. Crea un nuevo Notebook en Colab, asegúrate de cambiar el entorno a **R**.
2. Ejecuta el siguiente código para simular la heterocedasticidad y ver el poder diagnóstico de la función `plot()` aplicada a un modelo lineal.

```R
# Fijar semilla
set.seed(100)

# Simular heterocedasticidad
presion <- seq(10, 100, length.out=50)
error_het <- rnorm(50, mean=0, sd=presion*0.15)
deformacion <- 5.0 + 2.5 * presion + error_het

# Introducir el outlier influyente
presion[50] <- 150
deformacion[50] <- 100

datos_r <- data.frame(Presion=presion, Deformacion=deformacion)

# Ajustar el modelo
modelo_r <- lm(Deformacion ~ Presion, data=datos_r)

# Graficar diagnóstico (En R, plot(modelo) genera 4 gráficos clave)
# Nos enfocamos en el Gráfico 1 (Homocedasticidad) y el Gráfico 4 (Distancia de Cook)
par(mfrow=c(1,2))
plot(modelo_r, which=1, main="Residuos vs Ajustados")
plot(modelo_r, which=4, main="Distancia de Cook (Influencia)")

```

**Paso 2: Documentación en Posit Cloud y Despliegue en RPubs**

1. Abre tu espacio en **Posit Cloud** y crea un nuevo documento **RMarkdown**.
2. Transfiere el código de R y ejecútalo.
3. Escribe tu análisis técnico: *Basado en el gráfico de la Distancia de Cook que generaste en R, ¿qué nos indica el pico extremo del último punto de datos? Como ingeniero, si descubres que ese punto fue un error del sensor durante la prueba de laboratorio, ¿qué acción tomarías respecto al modelo?*
4. Compila (Knit) a HTML y publica en **RPubs**.

---